In [110]:
from langgraph.graph import StateGraph, START,END
from pydantic import BaseModel
from typing import TypedDict
from langgraph.types import Command, Send

In [111]:
class NodeState(BaseModel):
    tasks:list[str]=[]
    task:str =""

In [112]:
def planner(state:NodeState)->Command:
   tasks=['A','B','C']
   return Command(
         update={"tasks":tasks},
         goto=[Send("Worker",NodeState(tasks=tasks, task=t)) 
         for t in tasks]
      )
   
def worker(state:NodeState)->dict:
   print(f"Processingn----{state.task}")
   return {}
def merge(state:NodeState)->dict:
   return {}
    

In [113]:
graph=StateGraph(NodeState)
graph.add_node('Planner',planner)
graph.add_node('Worker',worker)
graph.add_node('Merge',merge)
graph.add_edge(START,'Planner')
graph.add_edge('Worker','Merge')
graph.add_edge('Merge',END)
app=graph.compile()
app.invoke({
    "tasks":[],
    "task":""
    })


Processingn----A
Processingn----B
Processingn----C


{'tasks': ['A', 'B', 'C'], 'task': ''}

In [115]:
from langgraph.graph import StateGraph, START,END
from pydantic import BaseModel
from typing import TypedDict
from langgraph.types import Command, Send

In [125]:
class NodeState(BaseModel):
    score:int

In [124]:
def decider(state:NodeState)->Command:
    if state.score>80:
            print("Approved Node")
            t=Send("Approved", NodeState(score=state.score))
    else:
        print("Rejected Node")
        t=Send("Reject", NodeState(score=state.score))
    return Command(
       update={"score":state.score},
       goto=t        
    )
def approved(state:NodeState)->dict:
    return {}
def reject(state:NodeState)->dict:
    return {}
graph=StateGraph(NodeState)
graph.add_node("Decider",decider)
graph.add_node("Approved",approved)
graph.add_node("Reject",reject)
graph.add_edge(START,"Decider")
graph.add_edge("Approved", END)        # <-- missing before
graph.add_edge("Reject", END)
app=graph.compile()
app.invoke({
    "score":30
})
print(app)

Rejected Node
